In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r"C:\Users\Administrator\Desktop\df_file.csv")
df.head()

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,0
1,Army chiefs in regiments decision\n \n Militar...,0
2,Howard denies split over ID cards\n \n Michael...,0
3,Observers to monitor UK election\n \n Minister...,0
4,Kilroy names election seat target\n \n Ex-chat...,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Text    2225 non-null   object
 1   Label   2225 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 34.9+ KB


In [11]:
print(df.columns)

Index(['Text', 'Label'], dtype='object')


In [12]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()

corpus = []
for i in range(len(df)):
    rp = re.sub('[^a-zA-Z]', ' ', df['Text'][i])   # fixed regex
    rp = rp.lower()
    rp = rp.split()
    
    rp = [wnl.lemmatize(word) for word in rp if word not in set(stopwords.words('english'))]
    
    rp = " ".join(rp)
    corpus.append(rp)

print(corpus)


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [13]:
from sklearn.feature_extraction.text import CountVectorizer
cv= CountVectorizer()
X= cv.fit_transform(corpus)

In [14]:
from sklearn.decomposition import LatentDirichletAllocation
model=LatentDirichletAllocation(n_components=4)
model.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",4
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [15]:
topic_results=model.transform(X)

In [16]:
topic_results[0]

array([8.43988815e-04, 8.55840471e-04, 9.97454258e-01, 8.45912712e-04])

In [17]:
topic_results[0].argmax()

np.int64(2)

In [18]:
df['group']=topic_results.argmax(axis=1)

In [19]:
df.head()

,Text,Label,group
0,Budget to set scene for election\n \n Gordon B...,0,2
1,Army chiefs in regiments decision\n \n Militar...,0,2
2,Howard denies split over ID cards\n \n Michael...,0,2
3,Observers to monitor UK election\n \n Minister...,0,2
4,Kilroy names election seat target\n \n Ex-chat...,0,0


In [22]:
# showing top words per topic
for index, topic in enumerate(model.components_):
    print(f"THE TOP 10 WORDS FOR TOPIC #{index}")
    print([cv.get_feature_names_out()[i] for i in topic.argsort()[-10:]])
    print("\n")

THE TOP 10 WORDS FOR TOPIC #0
['user', 'net', 'site', 'one', 'mail', 'firm', 'service', 'mr', 'people', 'said']


THE TOP 10 WORDS FOR TOPIC #1
['new', 'world', 'mobile', 'first', 'time', 'one', 'people', 'year', 'game', 'said']


THE TOP 10 WORDS FOR TOPIC #2
['minister', 'new', 'labour', 'also', 'bn', 'government', 'year', 'would', 'mr', 'said']


THE TOP 10 WORDS FOR TOPIC #3
['music', 'club', 'would', 'also', 'one', 'award', 'film', 'best', 'year', 'said']


